## 탐색적 데이터 분석(EDA)
- EDA(Exploratory Data Analysis) - 데이터의 구조적 특성을 규명하는 탐색 단계
- 미국 벨 연구소의 통계학자 '존 튜키(John Tukey)'가 창안한 데이터 해석 및 기획 절차
- 수치적 수집과 결과 도출에 앞서, 수집된 정보 자체에 대한 '다각적 관찰 및 직관적 파악'을 관통해야 함을 강조한다
- 정형화된 시각 외에 입체적인 시선으로 데이터 내부의 형태를 점검해야 한다
    - 1) 각 피처(Column)와 데이터 건수(Row)가 상징하는 메타 정보 및 속성 간 상호작용 검토
    - 2) 누락된 데이터(결측치) 가공 및 목적에 부합하는 레코드 선별(Filtering)
    - 3) 그래픽 요소를 통한 분포 및 특성 시각화

In [18]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 윈도우 맑은 고딕 설정
# plt.rcParams['font.family'] = 'Malgun Gothic'
# plt.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지

# 맥 애플고딕 설정
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지

## 타이타닉 데이터셋 컬럼 상세 정리

주피터 노트북에서 전처리를 시작하기 전, 각 변수(Feature)의 의미와 데이터 타입을 파악하기 위한 가이드라인입니다.

| 컬럼명 (Column) | 의미 (Description) | 데이터 타입 | 전처리 및 분석 팁 |
| :--- | :--- | :--- | :--- |
| **PassengerId** | 승객 고유 번호 | 수치형 (Integer) | 단순 일련번호이므로 예측 모델을 만들 때는 보통 **삭제(drop)**합니다. |
| **Survived** | 생존 여부 | 범주형 (Binary) | **0 = 사망, 1 = 생존**. 우리가 최종적으로 예측해야 하는 **목표 변수(Target)**입니다. |
| **Pclass** | 티켓 등급 (사회적 지위) | 범주형 (Ordinal) | **1 = 1등석, 2 = 2등석, 3 = 3등석**. 숫자로 되어 있지만 등급을 나타내는 범주형 데이터입니다. |
| **Name** | 승객 이름 | 문자열 (Text) | `Mr.`, `Mrs.`, `Miss.` 같은 **칭호(Title)를 추출**하여 파생 변수로 활용할 수 있습니다. |
| **Sex** | 성별 | 범주형 (Text) | `male`(남성), `female`(여성). 문자열이므로 머신러닝 모델 학습 전 **숫자(0 또는 1)로 인코딩**해야 합니다. |
| **Age** | 나이 | 수치형 (Continuous) | **결측치(NaN)가 존재**합니다. 전체 평균이나 칭호별 평균/중앙값으로 채우는 연습을 합니다. |
| **SibSp** | 동반한 형제 자매 / 배우자 수 | 수치형 (Discrete) | 함께 탑승한 형제, 자매, 배우자의 수입니다. 파생 변수를 만들 때 활용됩니다. |
| **Parch** | 동반한 부모 / 자식 수 | 수치형 (Discrete) | 함께 탑승한 부모, 자녀의 수입니다. `SibSp`와 더해 가족 크기를 계산할 수 있습니다. |
| **Ticket** | 티켓 번호 | 문자열 (Text) | 알파벳과 숫자가 섞인 무작위 문자열입니다. 정제하기 까다로워 보통 **삭제**합니다. |
| **Fare** | 여객 운임 (티켓 요금) | 수치형 (Continuous) | 데이터 분포가 한쪽으로 심하게 치우쳐(Skewed) 있어 **로그 변환(Log Transformation)**을 주로 적용합니다. |
| **Cabin** | 객실 번호 | 문자열 (Text) | **결측치가 70% 이상으로 매우 많습니다.** 아예 삭제하거나, 앞 글자(알파벳 구역)만 추출해 활용합니다. |
| **Embarked** | 탑승 항구명 | 범주형 (Text) | 승객이 배를 탄 항구입니다. (**C** = 셰르부르, **Q** = 퀸즈타운, **S** = 사우샘프턴). 결측치가 2개 존재합니다. |

In [19]:
base_path = r"/Users/kdt_0900_lyn/data_analysis/resource/dataset"
file_name = r"titanic.csv"

file_path = os.path.join(base_path, file_name)

In [20]:
pd.read_csv(file_path).head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 상관 관계
- ※ 열(Column), 데이터 속성(Attribute), 변수(Variable), 피처(Feature)는 모두 동의어로 혼용된다.
- 특정 수치 간의 상호 연결성, 즉 두 변수가 보여주는 변화 패턴의 유사성을 측정하는 지표이다. 특히 두 변수 사이의 관계가 선형적(Linear) 형태를 띠는지 평가하는 척도로 쓰인다.
- 예를 들어, A 변수의 수치가 상승할 때 B 변수의 수치 역시 동반 상승하는지, 아니면 반대로 하락하는지 파악한다.
- 수치의 증감 경향성을 확인할 수 있어, 전체적인 데이터의 분포 현황과 경향성(Trend)을 파악할 때 핵심적인 역할을 한다.
- 값의 범위는 -1에서 +1 사이의 수치로 산출된다.
- +1에 근접할수록 두 변수 A와 B가 동일한 방향으로 완벽하게 비례하여 움직인다.
- -1에 근접할수록 한 변수가 증가할 때 다른 변수는 감소하는 반비례 양상을 보인다.
- 0에 가까워질수록 두 변수 간에는 어떠한 선형적 연관성도 존재하지 않음을 뜻한다.
- 부호와 상관없이 절대값이 1에 가까워질수록 상관성의 정도가 강하다고 판단한다.
- 참고사항: 단, 높은 상관계수가 곧 두 변수 간의 인과관계를 의미하는 것은 아니다. 연관성이 높다는 사실이 반드시 원인과 결과 관계임을 보장하지는 않는다.

In [21]:
df.corr(numeric_only=True)

NameError: name 'df' is not defined

In [22]:
plt.matshow(df.corr(numeric_only=True))
plt.colorbar()
plt

NameError: name 'df' is not defined

## 데이터 전처리 

In [13]:
df.info()

NameError: name 'df' is not defined

In [ ]:
df.describe()

In [ ]:
df..unique()

In [ ]:
df.drop(["PassengerId", "Ticket", "Cabin", "Embarked"], )

## 계획 수립 
1. 분류형: 'PassengerID', 'Survived', 'Pclass', 'Name', 'Sex', 'Ticket', 'cabin', 'Embarked'
   - 최빈값 , 컬럼 생성 
3. 수치형: 'Age', 'SibSp', 'Parch', 'Fare'
       - 평균 , 중앙값 

## 이상치,  결측치, 중복값 확인 대체

In [ ]:
isna()

In [ ]:
age_median = df1["Age"].median()


In [ ]:
# 분위수
Q3 = df1["Age"].quantile(0.25)
Q1 = df1["Age"].quantile(0.75)
IQR = Q3 - Q1

max_ouylier = Q3 + (1.5 * IQR)
max_ouylier = Q1 - 

In [25]:
# 가장 작은 나이의 데이터 
df1.iloc[df1["Age"].idxmin()]["Age"]

NameError: name 'df1' is not defined

In [ ]:
df1.iloc[df1["Age"].idxmin(), "Age"] age_median
df2.iloc[803]

In [ ]:
df1["Age"] < 10

In [ ]:
df["Age"],isna().sum()

In [ ]:
df1["Age"] = df["Age"].fillna(age_median)

In [ ]:
df1.isna().sum()

In [ ]:
df1

In [26]:
df1["Age"] = df["Age"].astype(int)
df1.info()

NameError: name 'df' is not defined

### 가설1. 영화에서도 여성이 먼저 구출되었다. 실제 네이터도 여성의 생존률이 높지 않을까?

In [ ]:
print(df1.groupby("Sex")["Survived"].sum())


print(len(df1))

In [ ]:
print(df1.groupby("Sex")["Survived"].mean() * 100)

In [ ]:
df1.groupby("Sex")["Survived"].agg(["count", "mean", "sum"])

In [ ]:
sex_survived = df1.groupby("Sex")["Survived"].mean()

plt.figure(figsize=(8, 4))
plt.bar(sex_survived.index, sex_survived.values, color=["pink", "green"])
plt.ylim(0, 1)
plt.show()

# seaborn
- Matplotlib 위에 구축되어서 데이터를 더 간단하고 쉽게 시각화 할 수 있게 도와주는 데이터 시각화 라이브러리이다.
- 주로 통계적인 데이터 및 시각화에 초점이 맞춰져 있으며, 차트, 스타일을 제공한다.

[seaborn 공식링크](https://seaborn.pydata.org)

In [28]:
import seaborn as sns

In [ ]:

plt.figure(figsize=(4, 3))
plt.barplot(x="Sex", y="Survived", data=dfq, hue="Sex", palette="pastel")
plt.ylim(0, 1)
plt.show()

### 가설1 검증 
- 영화에서도 여성이 먼저 구출되었다. 실제 데이터도 여성의 생존률이 높지 않을까?
- 검증: 여성의 생존율이 74.2%로 남성의 생존율 18.9%보다 약 4배 가까운 정도로 압도적으로 높다 

In [ ]:
# 가설을 1개 세우고, 가설을 시각화하고 가설을 검증하여 결론을 도출하세요.

In [ ]:
# 가설 : 노인보다 , 어린 아이가 더 많이 구출됬을 것이다

In [ ]:
plt.figure(figsize=(4, 3))
plt.barplot(x="Age", y="Survived", data=dfq, hue="Age", palette="pastel")
plt.ylim(0, 1)
plt.show()